<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/options_flow_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import json
from datetime import datetime, date
import warnings
warnings.filterwarnings("ignore")
# ensure reproducibility
random.seed(42)
print("Libraries Installed!")

Libraries Installed!


In [27]:
# =============================================================
# GOOGLE DRIVE MOUNT
# =============================================================

def mount_drive():
    """
    Mount Google Drive in Colab.
    Run this once at the start of each session.
    """
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Google Drive mounted successfully.")
    except Exception as e:
        print(f"Drive mount failed: {e}")



## Configuration

In [28]:
# ── Change this folder path if you prefer a different location
DRIVE_CSV_PATH = "/content/drive/MyDrive/OptionsFlow/options_flow_history.csv"

# Score thresholds
STRONG_BUY_THRESHOLD  = 75
BUY_THRESHOLD         = 55
WATCH_THRESHOLD       = 40

# Week-over-week change thresholds for commentary
SIGNIFICANT_CHANGE    = 10   # composite score change considered meaningful
LARGE_FLOW_SHIFT      = 15   # flow score change considered large
TREND_SHIFT           = 25   # trend score change considered a trend shift

## # HISTORY MANAGER — Google Drive CSV

In [31]:

class FlowHistoryManager:
    """
    Saves and loads week-over-week flow data from a CSV
    stored on Google Drive. Handles mixed date formats cleanly.
    """

    def __init__(self, path: str = DRIVE_CSV_PATH):
        self.path = path
        self._ensure_folder()

    def _ensure_folder(self):
        """Create the folder on Drive if it doesn't exist yet."""
        folder = os.path.dirname(self.path)
        if folder and not os.path.exists(folder):
            os.makedirs(folder, exist_ok=True)
            print(f"Created folder: {folder}")

    # ── Load ──────────────────────────────────────────────────
    def load(self) -> pd.DataFrame:

        if not os.path.exists(self.path):
            return pd.DataFrame()

        df = pd.read_csv(self.path)

        if df.empty:
            return df

        # normalize run_date — strip any time component
        df["run_date"] = pd.to_datetime(
            df["run_date"], format="mixed"
        ).dt.normalize().dt.date

        return df

    # ── Save ──────────────────────────────────────────────────
    def save(self, records: list):

        if not records:
            return

        new_df = pd.DataFrame(records)

        # always store as plain date string
        new_df["run_date"] = pd.to_datetime(
            new_df["run_date"]
        ).dt.strftime("%Y-%m-%d")

        if os.path.exists(self.path):

            existing = pd.read_csv(self.path)
            existing["run_date"] = pd.to_datetime(
                existing["run_date"], format="mixed"
            ).dt.strftime("%Y-%m-%d")

            combined = pd.concat(
                [existing, new_df], ignore_index=True
            )

            # deduplicate — keep latest run per ticker per week
            combined["_week"] = pd.to_datetime(
                combined["run_date"]
            ).dt.to_period("W")

            combined = combined.drop_duplicates(
                subset=["ticker", "_week"], keep="last"
            ).drop(columns=["_week"])

        else:
            combined = new_df

        combined.to_csv(self.path, index=False)
        print(f"\nHistory saved → {self.path}")

    # ── Get Previous Week ─────────────────────────────────────
    def get_previous(self, ticker: str, current_run_date) -> dict:
        """
        Returns the most recent prior week record for a ticker.
        Returns None if no prior history exists.
        """

        df = self.load()

        if df.empty:
            return None

        current = pd.to_datetime(current_run_date).date()

        ticker_history = df[
            (df["ticker"] == ticker)
            & (pd.to_datetime(df["run_date"]).dt.date < current)
        ].sort_values("run_date", ascending=False)

        if ticker_history.empty:
            return None

        return ticker_history.iloc[0].to_dict()

    # ── Full History for One Ticker ───────────────────────────
    def ticker_history(self, ticker: str) -> pd.DataFrame:

        df = self.load()

        if df.empty:
            return pd.DataFrame()

        return df[df["ticker"] == ticker].sort_values(
            "run_date", ascending=False
        ).reset_index(drop=True)

In [32]:

# =============================================================
# COMMENTARY ENGINE
# =============================================================

class CommentaryEngine:
    """
    Generates signal + detailed narrative per stock.
    Includes week-over-week commentary when prior data exists.
    """

    def generate(self, ticker: str, current: dict, previous: dict = None):

        signal    = self._signal(current, previous)
        headline  = self._headline(ticker, signal, current, previous)
        narrative = self._narrative(ticker, current, previous)

        return {
            "signal":    signal,
            "headline":  headline,
            "narrative": narrative,
        }

    # ── Signal ────────────────────────────────────────────────
    def _signal(self, current, previous):

        composite = current["composite_score"]
        trend     = current["trend_score"]
        flow      = current["flow_score"]

        if previous:
            comp_delta = composite - previous["composite_score"]
            flow_delta = flow - previous["flow_score"]

            if comp_delta < -SIGNIFICANT_CHANGE and flow_delta < -LARGE_FLOW_SHIFT:
                return "SELL / REDUCE"

            if comp_delta < -SIGNIFICANT_CHANGE:
                return "WATCH — Weakening"

        if composite >= STRONG_BUY_THRESHOLD and trend >= 75 and flow >= 60:
            return "STRONG BUY"

        if composite >= BUY_THRESHOLD and trend >= 50:
            return "BUY"

        if composite >= WATCH_THRESHOLD:
            return "WATCH"

        if trend < 25 and flow < 25:
            return "AVOID"

        return "NEUTRAL"

    # ── Headline ──────────────────────────────────────────────
    def _headline(self, ticker, signal, current, previous):

        composite = current["composite_score"]
        direction = ""

        if previous:
            try:
                delta = composite - float(previous["composite_score"])
                if abs(delta) >= SIGNIFICANT_CHANGE:
                    direction = (
                        f" (+{delta:.1f})" if delta > 0
                        else f" ({delta:.1f})"
                    )
            except Exception:
                pass

        return (
            f"{ticker} | {signal} | "
            f"Composite: {composite:.1f}{direction} | "
            f"Trend: {current['trend_score']:.0f} | "
            f"Flow: {current['flow_score']:.0f}"
        )

    # ── Full Narrative ────────────────────────────────────────
    def _narrative(self, ticker, current, previous):

        lines = []

        c_comp = current["composite_score"]
        c_trend = current["trend_score"]
        c_flow  = current["flow_score"]
        c_bull  = current.get("bullish_ratio", 0.5)
        c_vol   = current.get("unusual_contracts", 0)
        c_prem  = current.get("total_premium", 0)
        c_pcr   = current.get("put_call_ratio", 1.0)
        c_mom   = current.get("momentum_3m", 0)

        # ── Trend
        if c_trend >= 75:
            lines.append(
                "Trend structure is strong — price is above both key SMAs "
                "with the longer average rising, confirming a healthy uptrend."
            )
        elif c_trend >= 50:
            lines.append(
                "Trend is constructive but not fully aligned. "
                "Price is holding above key moving averages but momentum needs confirmation."
            )
        elif c_trend >= 25:
            lines.append(
                "Trend structure is mixed. Price may be below one or more key moving averages. "
                "Risk of further weakness if support breaks."
            )
        else:
            lines.append(
                "Trend is broken. Price is below key moving averages and momentum "
                "is negative — avoid new long exposure."
            )

        # ── Momentum
        if c_mom > 0.10:
            lines.append(
                f"3-month momentum is strong at +{c_mom*100:.1f}%, "
                f"indicating sustained buying pressure."
            )
        elif c_mom > 0:
            lines.append(
                f"3-month momentum is modestly positive at +{c_mom*100:.1f}%."
            )
        elif c_mom < -0.10:
            lines.append(
                f"3-month momentum is deeply negative at {c_mom*100:.1f}% — "
                f"institutional sellers may be in control."
            )
        else:
            lines.append(
                f"3-month momentum is slightly negative at {c_mom*100:.1f}%."
            )

        # ── Options flow
        if c_bull >= 0.70:
            lines.append(
                f"Options flow is decisively bullish — call premium represents "
                f"{c_bull*100:.0f}% of total, indicating institutional upside positioning."
            )
        elif c_bull >= 0.55:
            lines.append(
                f"Options flow is leaning bullish with calls at "
                f"{c_bull*100:.0f}% of total premium."
            )
        elif c_bull <= 0.35:
            lines.append(
                f"Options flow is bearish — put premium dominates at "
                f"{(1-c_bull)*100:.0f}% of total. Smart money may be hedging or shorting."
            )
        else:
            lines.append(
                "Options flow is balanced — no clear directional bias this week."
            )

        # ── Put/call ratio
        if c_pcr < 0.5:
            lines.append(
                f"Put/call ratio of {c_pcr:.2f} is low — participants are "
                f"positioning aggressively for upside."
            )
        elif c_pcr > 1.5:
            lines.append(
                f"Put/call ratio of {c_pcr:.2f} is elevated — hedging or "
                f"outright bearish bets are rising."
            )

        # ── Unusual activity
        if c_vol > 20:
            lines.append(
                f"Unusual options activity is high with {c_vol} contracts "
                f"showing volume significantly above open interest — strong institutional footprint."
            )
        elif c_vol > 10:
            lines.append(
                f"{c_vol} contracts showed unusual volume — moderate institutional footprint."
            )
        else:
            lines.append("Unusual options activity is limited this week.")

        # ── Premium size
        if c_prem >= 10_000_000:
            lines.append(
                f"Total options premium of ${c_prem/1e6:.1f}M is institutional scale."
            )
        elif c_prem >= 5_000_000:
            lines.append(
                f"Total options premium of ${c_prem/1e6:.1f}M is meaningful."
            )
        elif c_prem > 0:
            lines.append(
                f"Total options premium of ${c_prem/1e6:.1f}M is modest — retail scale."
            )

        # ── Week-over-week
        if previous:
            try:
                comp_delta  = c_comp  - float(previous["composite_score"])
                flow_delta  = c_flow  - float(previous["flow_score"])
                trend_delta = c_trend - float(previous["trend_score"])
                prev_date   = previous.get("run_date", "prior week")

                wow_lines = [f"\nWEEK-OVER-WEEK (vs {prev_date}):"]

                if abs(comp_delta) >= SIGNIFICANT_CHANGE:
                    direction = "improved" if comp_delta > 0 else "deteriorated"
                    wow_lines.append(
                        f"Composite score {direction} by {abs(comp_delta):.1f} pts "
                        f"({float(previous['composite_score']):.1f} → {c_comp:.1f})."
                    )

                if abs(flow_delta) >= LARGE_FLOW_SHIFT:
                    direction = "surged" if flow_delta > 0 else "dropped"
                    wow_lines.append(
                        f"Options flow score {direction} by {abs(flow_delta):.1f} pts — "
                        f"significant shift in institutional positioning."
                    )
                elif abs(flow_delta) >= 5:
                    direction = "ticked up" if flow_delta > 0 else "eased"
                    wow_lines.append(
                        f"Flow score {direction} modestly by {abs(flow_delta):.1f} pts."
                    )

                if abs(trend_delta) >= TREND_SHIFT:
                    direction = "strengthened" if trend_delta > 0 else "weakened"
                    wow_lines.append(
                        f"Trend structure {direction} materially "
                        f"({float(previous['trend_score']):.0f} → {c_trend:.0f}) — "
                        f"{'bullish development.' if trend_delta > 0 else 'watch for further deterioration.'}"
                    )

                if len(wow_lines) == 1:
                    wow_lines.append(
                        "Scores are largely stable — no significant shift this week."
                    )

                lines.extend(wow_lines)

            except Exception:
                lines.append(
                    "\nWEEK-OVER-WEEK: Could not compute changes — "
                    "prior data format issue."
                )
        else:
            lines.append(
                "\nWEEK-OVER-WEEK: No prior week data available — "
                "this is the first recorded run for this ticker."
            )

        return " ".join(lines)


# =============================================================
# MAIN ENGINE
# =============================================================

class OptionsFlowEngine:
    """
    Institutional Options Flow Engine.
    Saves results to Google Drive CSV for week-over-week tracking.
    """

    def __init__(
        self,
        tickers:  list,
        csv_path: str = DRIVE_CSV_PATH
    ):
        self.tickers      = tickers
        self.price_data   = {}
        self.options_data = {}
        self.flow_scores  = []
        self.history      = FlowHistoryManager(csv_path)
        self.commentary   = CommentaryEngine()
        self.run_date     = date.today().isoformat()

    # =========================================================
    # 1. PRICE DATA
    # =========================================================
    def get_price_data(self):

        print("Fetching price data...")

        for ticker in self.tickers:
            try:
                stock = yf.Ticker(ticker)
                hist  = stock.history(period="1y")
                if isinstance(hist.columns, pd.MultiIndex):
                    hist.columns = hist.columns.get_level_values(0)
                self.price_data[ticker] = hist
            except Exception as e:
                print(f"  Price error {ticker}: {e}")

    # =========================================================
    # 2. OPTIONS DATA
    # =========================================================
    def get_options_data(self):

        print("Fetching options data...")

        for ticker in self.tickers:
            try:
                stock    = yf.Ticker(ticker)
                expiries = stock.options

                if not expiries:
                    continue

                all_options = []

                for expiry in expiries[:4]:
                    chain = stock.option_chain(expiry)

                    calls = chain.calls.copy()
                    puts  = chain.puts.copy()

                    calls["type"]   = "CALL"
                    puts["type"]    = "PUT"
                    calls["expiry"] = expiry
                    puts["expiry"]  = expiry

                    dte = (
                        pd.to_datetime(expiry) - pd.Timestamp.today()
                    ).days

                    calls["daysToExpiration"] = dte
                    puts["daysToExpiration"]  = dte

                    all_options.append(pd.concat([calls, puts]))

                if all_options:
                    self.options_data[ticker] = pd.concat(
                        all_options, ignore_index=True
                    )

            except Exception as e:
                print(f"  Options error {ticker}: {e}")

    # =========================================================
    # 3. TREND SCORE
    # =========================================================
    def calculate_trend_score(self, ticker):

        df    = self.price_data[ticker].copy()
        close = df["Close"]

        sma10w = close.rolling(50).mean()
        sma30w = close.rolling(150).mean()

        latest = close.iloc[-1]
        score  = 0

        if latest > sma10w.iloc[-1]:
            score += 20
        if latest > sma30w.iloc[-1]:
            score += 20

        sma30_slope = sma30w.iloc[-1] - sma30w.iloc[-10]
        if sma30_slope > 0:
            score += 20

        sma10_slope = sma10w.iloc[-1] - sma10w.iloc[-5]
        if sma10_slope > 0:
            score += 15

        if len(close) >= 63:
            mom = (latest / close.iloc[-63]) - 1
            if mom > 0.10:
                score += 25
            elif mom > 0:
                score += 15
            elif mom < -0.10:
                score -= 15

        return min(max(score, 0), 100)

    # =========================================================
    # 4. OPTIONS FLOW SCORE
    # =========================================================
    def calculate_flow_score(self, ticker):

        df = self.options_data[ticker].copy()

        df["volume"]       = pd.to_numeric(df["volume"],       errors="coerce").fillna(0)
        df["openInterest"] = pd.to_numeric(df["openInterest"], errors="coerce").fillna(0)
        df["lastPrice"]    = pd.to_numeric(df["lastPrice"],    errors="coerce").fillna(0)

        df["premium"] = df["lastPrice"] * df["volume"] * 100

        call_df       = df[df["type"] == "CALL"]
        put_df        = df[df["type"] == "PUT"]

        call_premium  = call_df["premium"].sum()
        put_premium   = put_df["premium"].sum()
        total_premium = call_premium + put_premium

        if total_premium == 0:
            return 0, {}

        bullish_ratio  = call_premium / total_premium
        call_volume    = call_df["volume"].sum()
        put_volume     = put_df["volume"].sum()
        put_call_ratio = put_volume / (call_volume + 1)

        df["vol_oi_ratio"] = df["volume"] / (df["openInterest"] + 1)
        unusual_count      = len(df[df["vol_oi_ratio"] > 2])

        score = 0

        if bullish_ratio >= 0.70:
            score += 30
        elif bullish_ratio >= 0.55:
            score += 15
        elif bullish_ratio <= 0.35:
            score -= 15

        if call_volume > put_volume * 1.5:
            score += 20
        elif put_volume > call_volume * 1.5:
            score -= 10

        if unusual_count > 20:
            score += 25
        elif unusual_count > 10:
            score += 15
        elif unusual_count > 5:
            score += 8

        if total_premium >= 10_000_000:
            score += 20
        elif total_premium >= 5_000_000:
            score += 12
        elif total_premium >= 1_000_000:
            score += 6

        short_dated = df[df["daysToExpiration"] < 7]
        if len(df) > 0 and len(short_dated) / len(df) > 0.5:
            score -= 15

        if put_call_ratio < 0.5:
            score += 10
        elif put_call_ratio > 1.5:
            score -= 10

        metrics = {
            "bullish_ratio":     round(bullish_ratio, 3),
            "put_call_ratio":    round(put_call_ratio, 3),
            "unusual_contracts": unusual_count,
            "total_premium":     round(total_premium, 2),
            "call_premium":      round(call_premium, 2),
            "put_premium":       round(put_premium, 2),
        }

        return min(max(score, 0), 100), metrics

    # =========================================================
    # 5. RUN ENGINE
    # =========================================================
    def run(self):

        self.get_price_data()
        self.get_options_data()

        records  = []
        rankings = []

        print(f"\nAnalyzing {len(self.tickers)} tickers...\n")
        print("=" * 70)

        for ticker in self.tickers:

            try:

                if ticker not in self.price_data:
                    continue
                if ticker not in self.options_data:
                    print(f"{ticker}: No options data — skipping")
                    continue

                trend_score         = self.calculate_trend_score(ticker)
                flow_score, metrics = self.calculate_flow_score(ticker)
                composite           = round(
                    (trend_score * 0.45) + (flow_score * 0.55), 2
                )

                close  = self.price_data[ticker]["Close"]
                mom_3m = float((close.iloc[-1] / close.iloc[-63]) - 1) \
                         if len(close) >= 63 else 0.0

                current = {
                    "composite_score": composite,
                    "trend_score":     trend_score,
                    "flow_score":      flow_score,
                    "momentum_3m":     round(mom_3m, 4),
                    **metrics,
                }

                # load previous week from Drive CSV
                previous = self.history.get_previous(ticker, self.run_date)

                commentary = self.commentary.generate(
                    ticker, current, previous
                )

                print(f"\n{'─'*70}")
                print(f"  {commentary['headline']}")
                print(f"{'─'*70}")
                print(f"  {commentary['narrative']}\n")

                records.append({
                    "run_date":          self.run_date,
                    "ticker":            ticker,
                    "composite_score":   composite,
                    "trend_score":       trend_score,
                    "flow_score":        flow_score,
                    "momentum_3m":       round(mom_3m * 100, 2),
                    "bullish_ratio":     metrics.get("bullish_ratio"),
                    "put_call_ratio":    metrics.get("put_call_ratio"),
                    "unusual_contracts": metrics.get("unusual_contracts"),
                    "total_premium":     metrics.get("total_premium"),
                    "signal":            commentary["signal"],
                    "narrative":         commentary["narrative"],
                })

                rankings.append({
                    "ticker":          ticker,
                    "signal":          commentary["signal"],
                    "composite_score": composite,
                    "trend_score":     trend_score,
                    "flow_score":      flow_score,
                    "momentum_3m_pct": round(mom_3m * 100, 2),
                    "bullish_ratio":   metrics.get("bullish_ratio"),
                    "unusual_count":   metrics.get("unusual_contracts"),
                    "total_premium_m": round(
                        metrics.get("total_premium", 0) / 1e6, 2
                    ),
                })

            except Exception as e:
                print(f"{ticker}: ERROR — {e}")
                continue

        # save to Google Drive
        if records:
            self.history.save(records)

        self.flow_scores = sorted(
            rankings, key=lambda x: x["composite_score"], reverse=True
        )

        return self.flow_scores

    # =========================================================
    # 6. TOP STOCKS
    # =========================================================
    def top_stocks(self, n=10):

        print(f"\n{'='*70}")
        print(f"  TOP {n} RANKED STOCKS")
        print(f"{'='*70}")
        print(
            f"{'#':<4}{'Ticker':<8}{'Signal':<22}"
            f"{'Composite':>10}{'Trend':>8}{'Flow':>8}"
            f"{'Mom%':>8}{'Bull%':>8}{'Prem$M':>9}"
        )
        print("─" * 85)

        for i, s in enumerate(self.flow_scores[:n], 1):
            bull_pct = round((s.get("bullish_ratio") or 0) * 100, 1)
            print(
                f"{i:<4}{s['ticker']:<8}{s['signal']:<22}"
                f"{s['composite_score']:>10.1f}"
                f"{s['trend_score']:>8.0f}"
                f"{s['flow_score']:>8.0f}"
                f"{s['momentum_3m_pct']:>8.1f}"
                f"{bull_pct:>8.1f}"
                f"{s['total_premium_m']:>9.2f}"
            )

    # =========================================================
    # 7. HISTORY REPORT — reads from Drive CSV
    # =========================================================
    def history_report(self, ticker: str):

        df = self.history.ticker_history(ticker)

        if df.empty:
            print(f"No history found for {ticker}.")
            return

        print(f"\n{'='*70}")
        print(f"  HISTORY REPORT: {ticker}")
        print(f"{'='*70}")
        print(
            f"{'Date':<14}{'Signal':<22}{'Composite':>10}"
            f"{'Trend':>8}{'Flow':>8}{'Mom%':>8}"
        )
        print("─" * 72)

        for _, row in df.iterrows():
            print(
                f"{str(row['run_date']):<14}"
                f"{str(row.get('signal', '')):<22}"
                f"{float(row['composite_score']):>10.1f}"
                f"{float(row['trend_score']):>8.0f}"
                f"{float(row['flow_score']):>8.0f}"
                f"{float(row.get('momentum_3m', 0)):>8.1f}"
            )


# =============================================================
# LONG FILTER
# =============================================================

def get_best_trades(watchlist=None, start="2022-01-01", top_n=10,
                    csv_path=DRIVE_CSV_PATH):

    if not watchlist:
        print("No watchlist provided.")
        return

    engine  = OptionsFlowEngine(watchlist, csv_path=csv_path)
    engine.get_price_data()
    engine.get_options_data()
    engine.run()

    results = pd.DataFrame(engine.flow_scores)

    if results.empty:
        print("No results.")
        return

    filtered = results[
        (results["signal"].isin(["STRONG BUY", "BUY"]))
        & (results["trend_score"] >= 60)
        & (results["flow_score"]  >= 50)
        & (results["momentum_3m_pct"] > 0)
        & (results["bullish_ratio"] >= 0.55)
    ].copy()

    filtered["rank_score"] = (
        filtered["composite_score"] * 0.40
        + filtered["trend_score"]   * 0.30
        + filtered["flow_score"]    * 0.30
    )

    filtered = filtered.sort_values(
        "rank_score", ascending=False
    ).head(top_n).reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  BEST LONG TRADES — Top {top_n}")
    print(f"{'='*60}\n")
    print(filtered.to_string(index=False))

    return filtered


# =============================================================
# SHORT FILTER
# =============================================================

def get_best_shorts(watchlist=None, start="2022-01-01", top_n=10,
                    csv_path=DRIVE_CSV_PATH):

    if not watchlist:
        print("No watchlist provided.")
        return

    engine = OptionsFlowEngine(watchlist, csv_path=csv_path)
    engine.get_price_data()
    engine.get_options_data()
    engine.run()

    results = pd.DataFrame(engine.flow_scores)

    if results.empty:
        print("No results.")
        return

    filtered = results[
        (results["signal"].isin(["SELL / REDUCE", "AVOID"]))
        & (results["trend_score"]     <= 30)
        & (results["momentum_3m_pct"] <  0)
        & (results["bullish_ratio"]   <= 0.40)
    ].copy()

    filtered["short_rank_score"] = (
        (100 - filtered["composite_score"]) * 0.40
        + (100 - filtered["trend_score"])   * 0.30
        + (100 - filtered["flow_score"])    * 0.30
    )

    filtered = filtered.sort_values(
        "short_rank_score", ascending=False
    ).head(top_n).reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  BEST SHORT CANDIDATES — Top {top_n}")
    print(f"{'='*60}\n")
    print(filtered.to_string(index=False))

    return filtered


In [34]:
# ── CELL 1: Mount Drive (run once per session)
mount_drive()


Mounted at /content/drive
Google Drive mounted successfully.


In [50]:
# ── CELL 2: Define your daily watchlist
watchlist = [
    "NVDA", "META", "AVGO", "MSFT", "GLW", "WDC","DHR" , "PODD" ,"LOW", "MOS", "VLTO" ,
    "AAPL", "AMZN", "TSLA", "JPM", "V", "PLTR", "NKE", "TECH", "NCLH"
]


In [51]:
# ── CELL 3: Run engine
engine   = OptionsFlowEngine(watchlist)
rankings = engine.run()

Fetching price data...
Fetching options data...

Analyzing 20 tickers...


──────────────────────────────────────────────────────────────────────
  NVDA | STRONG BUY | Composite: 86.2 | Trend: 100 | Flow: 75
──────────────────────────────────────────────────────────────────────
  Trend structure is strong — price is above both key SMAs with the longer average rising, confirming a healthy uptrend. 3-month momentum is strong at +21.8%, indicating sustained buying pressure. Options flow is leaning bullish with calls at 70% of total premium. Put/call ratio of 0.46 is low — participants are positioning aggressively for upside. Unusual options activity is high with 126 contracts showing volume significantly above open interest — strong institutional footprint. Total options premium of $430.2M is institutional scale. 
WEEK-OVER-WEEK: No prior week data available — this is the first recorded run for this ticker.


──────────────────────────────────────────────────────────────────────
  META | 

In [52]:
# ── CELL 4: View ranked table
engine.top_stocks(n=9)


  TOP 9 RANKED STOCKS
#   Ticker  Signal                 Composite   Trend    Flow    Mom%   Bull%   Prem$M
─────────────────────────────────────────────────────────────────────────────────────
1   AMZN    STRONG BUY                  89.0     100      80    31.3    76.8    35.88
2   NVDA    STRONG BUY                  86.2     100      75    21.8    69.6   430.20
3   AAPL    STRONG BUY                  86.2     100      75    13.9    69.8    87.78
4   GLW     STRONG BUY                  80.8     100      65    47.2    72.6    12.82
5   WDC     STRONG BUY                  78.0     100      60    69.7    66.3    27.39
6   V       BUY                         75.8      50      97     2.2    87.3     7.75
7   MSFT    BUY                         72.0      50      90     6.6    84.5   151.68
8   AVGO    BUY                         61.5     100      30    28.1    51.1    57.51
9   PLTR    WATCH                       56.2      15      90     0.7    62.4    47.73


In [53]:
# ── CELL 5: Best longs
get_best_trades(watchlist=watchlist, top_n=5)

Fetching price data...
Fetching options data...
Fetching price data...
Fetching options data...

Analyzing 20 tickers...


──────────────────────────────────────────────────────────────────────
  NVDA | STRONG BUY | Composite: 86.2 | Trend: 100 | Flow: 75
──────────────────────────────────────────────────────────────────────
  Trend structure is strong — price is above both key SMAs with the longer average rising, confirming a healthy uptrend. 3-month momentum is strong at +21.8%, indicating sustained buying pressure. Options flow is leaning bullish with calls at 70% of total premium. Put/call ratio of 0.46 is low — participants are positioning aggressively for upside. Unusual options activity is high with 126 contracts showing volume significantly above open interest — strong institutional footprint. Total options premium of $430.2M is institutional scale. 
WEEK-OVER-WEEK: No prior week data available — this is the first recorded run for this ticker.


────────────────────────────────

,ticker,signal,composite_score,trend_score,flow_score,momentum_3m_pct,bullish_ratio,unusual_count,total_premium_m,rank_score
0,AMZN,STRONG BUY,89.00,100,80,31.31,0.768,58,35.88,89.6
1,NVDA,STRONG BUY,86.25,100,75,21.82,0.696,126,430.20,87.0
2,AAPL,STRONG BUY,86.25,100,75,13.88,0.698,82,87.78,87.0
3,GLW,STRONG BUY,80.75,100,65,47.23,0.726,15,12.82,81.8
4,WDC,STRONG BUY,78.00,100,60,69.74,0.663,39,27.39,79.2


In [54]:
#─ CELL 6: Best shorts
get_best_shorts(watchlist=watchlist, top_n=5)


Fetching price data...
Fetching options data...
Fetching price data...
Fetching options data...

Analyzing 20 tickers...


──────────────────────────────────────────────────────────────────────
  NVDA | STRONG BUY | Composite: 86.2 | Trend: 100 | Flow: 75
──────────────────────────────────────────────────────────────────────
  Trend structure is strong — price is above both key SMAs with the longer average rising, confirming a healthy uptrend. 3-month momentum is strong at +21.8%, indicating sustained buying pressure. Options flow is leaning bullish with calls at 70% of total premium. Put/call ratio of 0.46 is low — participants are positioning aggressively for upside. Unusual options activity is high with 126 contracts showing volume significantly above open interest — strong institutional footprint. Total options premium of $430.2M is institutional scale. 
WEEK-OVER-WEEK: No prior week data available — this is the first recorded run for this ticker.


────────────────────────────────

,ticker,signal,composite_score,trend_score,flow_score,momentum_3m_pct,bullish_ratio,unusual_count,total_premium_m,short_rank_score
0,DHR,AVOID,0.0,0,0,-21.37,0.210,15,0.76,100.00
1,TECH,AVOID,0.0,0,0,-25.98,0.320,1,0.18,100.00
2,NCLH,AVOID,0.0,0,0,-35.60,0.277,13,0.54,100.00
3,PODD,AVOID,3.3,0,6,-40.14,0.162,12,1.25,96.88


In [43]:
# ── CELL 7: History for a specific ticker
engine.history_report("NVDA")


  HISTORY REPORT: NVDA
Date          Signal                 Composite   Trend    Flow    Mom%
────────────────────────────────────────────────────────────────────────
2026-05-16    STRONG BUY                  86.2     100      75    21.8
